# Experimento de interpolação temporal

AST e AudioMAE: 198 → 1024 quadros; PaSST: ~200 → 998. Interpolação bicúbica somente no tempo, após a normalização, inspirada no código oficial do HTS-AT. O dobramento específico do Swin não é aplicado aos outros modelos.

Comparação pareada nos quatro cenários e 13 classes, incluindo TS9. Novos controles e variantes usam os mesmos dados, checkpoint, precisão, atenção e protocolo de cabeça. Resultados anteriores são preservados; sem fine-tuning.


In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'configs').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'configs/linear_probe/temporal_interpolation.json').is_file():
    raise FileNotFoundError('Execute a partir de C:/TCC ou C:/TCC/notebooks')
sys.path.insert(0, str(ROOT))
RUN_PREVIEW = False
RUN_PREFLIGHT = False
RUN_EXTRACTION = False
RUN_TRAINING = False
plan = json.loads((ROOT / 'configs/linear_probe/temporal_interpolation.json').read_text())
print(json.dumps(plan, ensure_ascii=False, indent=2))


## Visualizar uma entrada real

Habilite somente `RUN_PREVIEW` para comparar os mesmos valores log-Mel antes e depois. A faixa de cores é comum às imagens. O eixo horizontal mostra posições da imagem; o alongamento não cria um WAV mais longo.


In [ ]:
if RUN_PREVIEW:
    import torch
    import matplotlib.pyplot as plt
    from src.data.archive import GuitarFxArchive
    from src.data.spectrogram_resize import adapt_spectrogram_time

    with GuitarFxArchive('mono_disc', 'mel16') as archive:
        effect, name = next(archive.iter_samples())
        original = torch.from_numpy(archive.read_feature(effect, name))
    original = (original + 4.27) / (4.57 * 2)
    padded = adapt_spectrogram_time(original, 1024, mode='pad')
    stretched = adapt_spectrogram_time(original, 1024)
    fig, axes = plt.subplots(3, 1, figsize=(12, 8), constrained_layout=True)
    vmin = min(0.0, float(original.min()), float(stretched.min()))
    vmax = max(float(original.max()), float(stretched.max()))
    for ax, value, title in zip(axes, (original, padded, stretched), ('Original: 198 quadros', 'Controle: preenchimento até 1024', 'Variante: alongamento até 1024')):
        im = ax.imshow(value.T, origin='lower', aspect='auto', vmin=vmin, vmax=vmax, cmap='magma')
        ax.set_title(title)
        ax.set_xlabel('Posição no espectrograma (quadro)')
        ax.set_ylabel('Banda Mel')
    fig.colorbar(im, ax=axes, label='Valor normalizado')
    plt.show()
else:
    print('Prévia desativada. Nenhum dataset foi lido.')


## Entradas e checkpoints de AST/AudioMAE

O coordenador lê os mel16 diretamente dos ZIPs e exige correspondência exata por efeito/nome com os manifests originais. Confere CRC e SHA-256 por feature, sem materializar a coleção inteira. A normalização histórica `(x - (-4.27)) / (4.57 * 2)` é mantida; padding ou interpolação acontece depois dela.

Os encoders são carregados com verificação estrita de todos os pesos usados. AST utiliza a conversão publicada do checkpoint AudioSet 0.4593, com SHA-256 fixado, e AudioMAE utiliza o checkpoint oficial local. O ambiente separado com timm 0.4.5 preserva o ambiente original. A implementação SDPA mantém a equação de atenção e foi comparada numericamente com a original em float32.

Preparação dos pesos: `python scripts/prepare_temporal_checkpoints.py`. Ambiente AST/AudioMAE: `python scripts/setup_temporal_runtime.py --prepare`, executado pelo Python de `.venv-transfer`; sem `--prepare`, o comando apenas inspeciona. As dependências estão em `requirements-temporal-legacy.txt`. Não altere o ambiente durante uma execução ativa.


In [ ]:
def make_interpolated_dataset(root):
    from src.data.dataset_ast import FxDatasetAST
    dataset = FxDatasetAST(
        root=str(root), target_length=1024, temporal_adaptation='bicubic',
        excl_folders=['MT2', '_NoFX_mono', '_NoFX_mono_preprocessed', '_NoFX_poly', '_NoFX_poly_preprocessed'],
    )
    dataset.init_dataset()
    return dataset

# AST: embeddings = frozen_ast(inputs)                # B,T,F
# AudioMAE: embeddings = frozen_audiomae(inputs.unsqueeze(1))  # B,1,T,F
# Saídas novas devem ir para plan['output_root'], em caches isolados.


## Integração com PaSST e verificação curta

O controle mantém o frontend nativo e a imagem curta (~200 quadros). A variante interpola somente o tempo para 998, mantendo 128 bandas, os pesos e a normalização. Ambos usam a mesma implementação SDPA e precisão BF16 nesta comparação.

Habilitar RUN_PREFLIGHT executa os checks curtos dos três encoders nos ambientes apropriados. Os benchmarks incluem inferência aquecida; não extraem caches nem treinam cabeças.


In [ ]:
if RUN_PREFLIGHT:
    import subprocess
    for model in ('passt', 'ast', 'audiomae'):
        env = '.venv-transfer' if model == 'passt' else '.venv-temporal-legacy'
        python = ROOT / env / 'Scripts/python.exe'
        subprocess.run([str(python), '-X', 'utf8', str(ROOT/'scripts/run_temporal_interpolation.py'), '--worker', 'preflight', '--model', model], cwd=ROOT, check=True)
else:
    print('Verificação técnica desativada.')


## Execução autorizada e comparação

O runner cria caches isolados e oferece retomada. Cada modelo/condição/cenário recebe oito candidatos na seed 42, dois finalistas com seeds 7 e 21, e cinco confirmações novas (101, 202, 303, 404, 505). São 408 novas cabeças, incluindo os controles; 120 confirmações finais. Seleção somente por F1 macro de validação.

O teste já foi consultado anteriormente: comparação exploratória. A alteração da escala dos mel16 é uma hipótese separada e não é combinada com este experimento. O HTS-AT não é reexecutado.


In [ ]:
if RUN_EXTRACTION != RUN_TRAINING:
    raise ValueError('Habilite as duas flags para o experimento completo; a consulta mantém ambas False.')
if RUN_EXTRACTION and RUN_TRAINING:
    import subprocess
    python = ROOT / '.venv-transfer/Scripts/python.exe'
    subprocess.run([str(python), '-X', 'utf8', str(ROOT/'scripts/run_temporal_interpolation.py'), '--run'], cwd=ROOT, check=True)
    subprocess.run([str(python), '-X', 'utf8', str(ROOT/'scripts/report_temporal_interpolation.py')], cwd=ROOT, check=True)
else:
    print('Extração e treinamento desativados. Apenas consulta de resultados existentes.')


## Consulta dos resultados

Relatório científico e JSON em `results/temporal_interpolation/`. Caches, heads, predições e logs ficam em `results/transfer_learning/ablations/temporal_interpolation_v1/` e `data/audits/transfer_learning/`, fora do Git. As matrizes são médias normalizadas por classe nas cinco seeds, com métricas calculadas por seed antes da agregação.


In [ ]:
summary_path = ROOT / 'results/temporal_interpolation/summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    import pandas as pd
    display(pd.DataFrame(summary['comparison_table']))
else:
    print('A consolidação completa ainda não foi publicada; consulte o estado local da execução.')
